# 📊 Customer & Orders — Full EDA + Advanced Analysis
**Datasets:** `customers.csv` (50,000 rows) · `orders.csv` (50,000 rows)  
**Sections:** Data Cleaning → EDA → Advanced Analysis (RFM, Forecasting, Churn)

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Style
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (12, 5)

print('Libraries loaded ✅')

---
## 📥 Load Data

In [ ]:
customers = pd.read_csv('customers.csv')
orders    = pd.read_csv('orders.csv')

print(f'customers → {customers.shape[0]:,} rows × {customers.shape[1]} cols')
print(f'orders    → {orders.shape[0]:,} rows × {orders.shape[1]} cols')

---
## 🧹 Section 1 — Data Cleaning

### 1.1 — Null Values

In [ ]:
print('=== CUSTOMERS — Null Counts ===')
cust_nulls = customers.isnull().sum()
print(cust_nulls[cust_nulls > 0] if cust_nulls.sum() > 0 else 'No nulls found ✅')

print('\n=== ORDERS — Null Counts ===')
ord_nulls = orders.isnull().sum()
print(ord_nulls[ord_nulls > 0])

In [ ]:
# Visualise missing data
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, df, title in zip(axes, [customers, orders], ['Customers', 'Orders']):
    missing_pct = df.isnull().mean() * 100
    missing_pct = missing_pct[missing_pct > 0]
    if missing_pct.empty:
        ax.text(0.5, 0.5, 'No missing values', ha='center', va='center', fontsize=13)
    else:
        missing_pct.sort_values().plot(kind='barh', ax=ax, color='coral')
        ax.set_xlabel('% Missing')
    ax.set_title(f'{title} — Missing %')
plt.tight_layout()
plt.show()

### 1.2 — Duplicates

In [ ]:
print(f'Customer duplicate rows : {customers.duplicated().sum()}')
print(f'Customer duplicate IDs  : {customers["customer_id"].duplicated().sum()}')
print(f'Order duplicate rows    : {orders.duplicated().sum()}')
print(f'Order duplicate IDs     : {orders["order_id"].duplicated().sum()}')

### 1.3 — Fix Data Types

In [ ]:
# Customers
customers['date_of_birth']     = pd.to_datetime(customers['date_of_birth'])
customers['registration_date'] = pd.to_datetime(customers['registration_date'])
customers['last_login_date']   = pd.to_datetime(customers['last_login_date'])
customers['membership_tier']   = pd.Categorical(
    customers['membership_tier'], categories=['Bronze','Silver','Gold','Platinum'], ordered=True
)
customers['account_status']    = customers['account_status'].astype('category')

# Orders
orders['order_date']    = pd.to_datetime(orders['order_date'])
orders['delivery_date'] = pd.to_datetime(orders['delivery_date'])
orders['category']      = orders['category'].astype('category')
orders['order_status']  = orders['order_status'].astype('category')

# Derived
snapshot = pd.Timestamp('2026-01-01')
customers['age'] = ((snapshot - customers['date_of_birth']).dt.days / 365.25).astype(int)
orders['order_year']  = orders['order_date'].dt.year
orders['order_month'] = orders['order_date'].dt.to_period('M')

print('Data types fixed ✅')
print(customers[['date_of_birth','registration_date','membership_tier','age']].dtypes)

### 1.4 — Outlier Detection (Box Plots)

In [ ]:
num_cols = ['grand_total_usd', 'unit_price_usd', 'quantity', 'discount_percent', 'customer_review_score']

fig, axes = plt.subplots(1, len(num_cols), figsize=(18, 5))
for ax, col in zip(axes, num_cols):
    orders[col].dropna().plot(kind='box', ax=ax, patch_artist=True,
                              boxprops=dict(facecolor='steelblue', alpha=0.6))
    ax.set_title(col, fontsize=10)
plt.suptitle('Outlier Detection — Orders Numeric Columns', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# IQR summary
for col in ['grand_total_usd', 'unit_price_usd']:
    Q1, Q3 = orders[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    out = orders[(orders[col] < Q1 - 1.5*IQR) | (orders[col] > Q3 + 1.5*IQR)]
    print(f'{col}: {len(out):,} outliers ({len(out)/len(orders)*100:.1f}%)')

---
## 🔍 Section 2 — Exploratory Data Analysis

### 2.1 — Age-wise Customer Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(customers['age'].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(customers['age'].mean(), color='red', linestyle='--', label=f'Mean: {customers["age"].mean():.1f}')
ax.axvline(customers['age'].median(), color='orange', linestyle='--', label=f'Median: {customers["age"].median():.1f}')
ax.set_xlabel('Age')
ax.set_ylabel('Number of Customers')
ax.set_title('Age Distribution of Customers')
ax.legend()
plt.tight_layout()
plt.show()

print(customers['age'].describe().round(1))

### 2.2 — Revenue Distribution (Skewness)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw
axes[0].hist(orders['grand_total_usd'].dropna(), bins=50, color='coral', edgecolor='white', alpha=0.85)
axes[0].set_title(f'Revenue Distribution  (skew={orders["grand_total_usd"].skew():.2f})')
axes[0].set_xlabel('Grand Total (USD)')

# Log-transformed
log_rev = np.log1p(orders['grand_total_usd'].dropna())
axes[1].hist(log_rev, bins=50, color='mediumseagreen', edgecolor='white', alpha=0.85)
axes[1].set_title(f'Log-Revenue Distribution  (skew={log_rev.skew():.2f})')
axes[1].set_xlabel('log(Grand Total + 1)')

plt.tight_layout()
plt.show()

### 2.3 — Correlation Matrix

In [ ]:
corr_cols = ['quantity','unit_price_usd','discount_percent','discount_amount_usd',
             'subtotal_usd','tax_amount_usd','shipping_fee_usd','grand_total_usd','customer_review_score']
corr = orders[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix — Numeric Order Features')
plt.tight_layout()
plt.show()

### 2.4 — Category-wise Sales (Bar Chart)

In [ ]:
cat_sales = orders.groupby('category')['grand_total_usd'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(cat_sales.index, cat_sales.values, color=sns.color_palette('muted', len(cat_sales)))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
for bar, val in zip(bars, cat_sales.values):
    ax.text(val + cat_sales.max()*0.01, bar.get_y() + bar.get_height()/2,
            f'${val/1e6:.2f}M', va='center', fontsize=9)
ax.set_title('Total Revenue by Category')
ax.set_xlabel('Revenue (USD)')
plt.tight_layout()
plt.show()

### 2.5 — Monthly Revenue Trend

In [ ]:
monthly = (orders.groupby('order_month')['grand_total_usd']
           .sum().reset_index())
monthly['order_month_dt'] = monthly['order_month'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['order_month_dt'], monthly['grand_total_usd'], marker='o', linewidth=2, color='steelblue')
ax.fill_between(monthly['order_month_dt'], monthly['grand_total_usd'], alpha=0.15, color='steelblue')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
ax.set_title('Monthly Revenue Trend')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 2.6 — State-wise Sales Map (Choropleth)

In [ ]:
# US state name → abbreviation mapping
state_abbrev = {
    'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA',
    'Colorado':'CO','Connecticut':'CT','Delaware':'DE','Florida':'FL','Georgia':'GA',
    'Hawaii':'HI','Idaho':'ID','Illinois':'IL','Indiana':'IN','Iowa':'IA',
    'Kansas':'KS','Kentucky':'KY','Louisiana':'LA','Maine':'ME','Maryland':'MD',
    'Massachusetts':'MA','Michigan':'MI','Minnesota':'MN','Mississippi':'MS',
    'Missouri':'MO','Montana':'MT','Nebraska':'NE','Nevada':'NV','New Hampshire':'NH',
    'New Jersey':'NJ','New Mexico':'NM','New York':'NY','North Carolina':'NC',
    'North Dakota':'ND','Ohio':'OH','Oklahoma':'OK','Oregon':'OR','Pennsylvania':'PA',
    'Rhode Island':'RI','South Carolina':'SC','South Dakota':'SD','Tennessee':'TN',
    'Texas':'TX','Utah':'UT','Vermont':'VT','Virginia':'VA','Washington':'WA',
    'West Virginia':'WV','Wisconsin':'WI','Wyoming':'WY','District of Columbia':'DC'
}

state_rev = (orders.groupby('shipping_state')['grand_total_usd']
             .sum().reset_index())
state_rev['abbrev'] = state_rev['shipping_state'].map(state_abbrev)
state_rev = state_rev.dropna(subset=['abbrev'])

fig = px.choropleth(
    state_rev,
    locations='abbrev',
    locationmode='USA-states',
    color='grand_total_usd',
    scope='usa',
    color_continuous_scale='Blues',
    labels={'grand_total_usd': 'Revenue (USD)'},
    title='State-wise Total Revenue'
)
fig.show()

### 2.7 — Review Score Distribution

In [ ]:
score_counts = orders['customer_review_score'].dropna().value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(score_counts.index.astype(str), score_counts.values,
              color=['#d73027','#f46d43','#fdae61','#74add1','#4575b4'])
for bar, val in zip(bars, score_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, f'{val:,}',
            ha='center', fontsize=9)
ax.set_title('Customer Review Score Distribution')
ax.set_xlabel('Score')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()
print(f'Average review score: {orders["customer_review_score"].mean():.2f}')

---
## 🚀 Section 3 — Advanced Analysis

### 3.1 — RFM Segmentation

In [ ]:
snapshot = pd.Timestamp('2026-01-01')

rfm = (orders[orders['order_status'] == 'Delivered']
       .groupby('customer_id')
       .agg(
           Recency  = ('order_date', lambda x: (snapshot - x.max()).days),
           Frequency= ('order_id',   'count'),
           Monetary = ('grand_total_usd', 'sum')
       ).reset_index())

# Score 1-5 (5=best)
rfm['R_score'] = pd.qcut(rfm['Recency'],   5, labels=[5,4,3,2,1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['Monetary'],  5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_Score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

def segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4 and m >= 4: return 'Champions'
    elif r >= 3 and f >= 3:           return 'Loyal Customers'
    elif r >= 4 and f <= 2:           return 'New Customers'
    elif r <= 2 and f >= 3:           return 'At Risk'
    elif r == 1 and f == 1:           return 'Lost'
    else:                             return 'Potential Loyalists'

rfm['Segment'] = rfm.apply(segment, axis=1)

seg_counts = rfm['Segment'].value_counts()
print(seg_counts)
print(f'\nTotal customers in RFM: {len(rfm):,}')

In [ ]:
# RFM Segment visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

seg_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2', len(seg_counts)), edgecolor='white')
axes[0].set_title('RFM Segment Distribution')
axes[0].set_ylabel('Customer Count')
axes[0].tick_params(axis='x', rotation=30)

seg_revenue = rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False)
seg_revenue.plot(kind='bar', ax=axes[1], color=sns.color_palette('Set1', len(seg_revenue)), edgecolor='white')
axes[1].set_title('RFM Segment — Total Revenue')
axes[1].set_ylabel('Revenue (USD)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# RFM 3D scatter
fig = px.scatter_3d(
    rfm.sample(3000, random_state=42),
    x='Recency', y='Frequency', z='Monetary',
    color='Segment', opacity=0.65,
    title='RFM 3D Scatter — Customer Segments'
)
fig.show()

### 3.2 — Sales Forecasting (Prophet)

In [ ]:
from prophet import Prophet

# Prepare daily revenue series
daily = (orders.groupby('order_date')['grand_total_usd']
         .sum().reset_index()
         .rename(columns={'order_date':'ds','grand_total_usd':'y'}))

model = Prophet(yearly_seasonality=True, weekly_seasonality=True,
                changepoint_prior_scale=0.05)
model.fit(daily)

future   = model.make_future_dataframe(periods=180)  # 6-month forecast
forecast = model.predict(future)

print('Model trained ✅')
print(forecast[['ds','yhat','yhat_lower','yhat_upper']].tail())

In [ ]:
fig = model.plot(forecast, figsize=(14, 5))
plt.title('Revenue Forecast — Next 6 Months (Prophet)')
plt.xlabel('Date')
plt.ylabel('Daily Revenue (USD)')
plt.tight_layout()
plt.show()

fig2 = model.plot_components(forecast)
plt.tight_layout()
plt.show()

### 3.3 — Churn Prediction (Logistic Regression)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, RocCurveDisplay)

# Label: churn = no order in last 180 days from snapshot
last_order = orders.groupby('customer_id')['order_date'].max().reset_index()
last_order['days_since'] = (snapshot - last_order['order_date']).dt.days
last_order['churned'] = (last_order['days_since'] > 180).astype(int)

# Features from RFM
churn_df = rfm.merge(last_order[['customer_id','churned']], on='customer_id')
churn_df = churn_df.merge(
    customers[['customer_id','age','membership_tier','account_status','email_subscribed']],
    on='customer_id', how='left'
)

churn_df['tier_code']    = churn_df['membership_tier'].cat.codes
churn_df['status_code']  = churn_df['account_status'].cat.codes
churn_df['email_int']    = churn_df['email_subscribed'].astype(int)

features = ['Recency','Frequency','Monetary','R_score','F_score','M_score',
            'age','tier_code','status_code','email_int']
churn_df = churn_df.dropna(subset=features + ['churned'])

X = churn_df[features]
y = churn_df['churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_sc, y_train)

y_pred = clf.predict(X_test_sc)
y_prob = clf.predict_proba(X_test_sc)[:,1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Active','Churned']))
print(f'ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Active','Churned'], yticklabels=['Active','Churned'])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC Curve
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1], name='Logistic Regression')
axes[1].set_title('ROC Curve — Churn Prediction')
axes[1].plot([0,1],[0,1], 'k--', alpha=0.4)

plt.tight_layout()
plt.show()

# Feature importances
coef_df = pd.DataFrame({'Feature': features, 'Coefficient': clf.coef_[0]})
coef_df = coef_df.sort_values('Coefficient')

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['red' if c > 0 else 'steelblue' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Churn Prediction — Feature Coefficients')
plt.tight_layout()
plt.show()

### 3.4 — Category Revenue Heatmap (Month × Category)

In [ ]:
heatmap_data = (orders.groupby(['order_month','category'])['grand_total_usd']
                .sum().unstack(fill_value=0))
heatmap_data.index = heatmap_data.index.astype(str)

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    heatmap_data / 1e3,
    cmap='YlOrRd', linewidths=0.4, linecolor='white',
    fmt='.0f', annot=True, annot_kws={'size': 7}, ax=ax
)
ax.set_title('Monthly Revenue by Category (USD thousands)')
ax.set_xlabel('Category')
ax.set_ylabel('Month')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 📋 Summary

| Section | Key Insights |
|---------|-------------|
| Data Cleaning | Nulls mainly in `delivery_date` & `customer_review_score` (pending/cancelled orders); no duplicates |
| Age Analysis | Broad spread; majority of customers are 30–55 years old |
| Revenue | Right-skewed; log transform normalises well |
| RFM | Champions & Loyal Customers drive the majority of revenue |
| Forecasting | Prophet captures yearly + weekly seasonality; 6-month projection available |
| Churn Model | Recency is the strongest churn predictor; ROC-AUC reported above |